# S08 — PyTorch in Practice

**Week 5 · Mon Sep 21, 2026 · Module 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s08_pytorch_in_practice.ipynb)

Every cell below is a worked example from the [S08 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s08/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s08.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s08.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## nn.Module: parameters that manage themselves


*Expected output starts with:* `x(4, 3) @ w(3, 2) + b(2,) -> (4, 2), dtype=torch.float32`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# --- tensors: shapes, dtypes, broadcasting ---
x = torch.randn(4, 3)                 # 4 samples, 3 features
w = torch.randn(3, 2)                 # a 3->2 linear map
b = torch.randn(2)                    # bias, broadcast over the batch
out = x @ w + b                       # (4,3) @ (3,2) + (2,) -> (4,2)
print(f"x{tuple(x.shape)} @ w{tuple(w.shape)} + b{tuple(b.shape)} -> {tuple(out.shape)}, dtype={out.dtype}")

col = x.mean(dim=0)                   # reduce over the batch dimension
print(f"mean over dim=0 -> shape {tuple(col.shape)}")

# --- autograd on tensors: same idea as Value, elementwise ---
w.requires_grad_(True)
loss = ((x @ w) ** 2).mean()
loss.backward()
print(f"loss = {loss.item():.4f}, w.grad shape = {tuple(w.grad.shape)}")

# --- nn.Module: parameters are registered automatically ---
class TinyMLP(nn.Module):
    def __init__(self, d_in, d_hidden, d_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

model = TinyMLP(3, 16, 2)
total = 0
for name, p in model.named_parameters():
    print(f"{name:12s} shape={tuple(p.shape)!s:10s} requires_grad={p.requires_grad}")
    total += p.numel()
print(f"total trainable parameters: {total}")

## The canonical training loop


*Expected output starts with:* `epoch  30: train loss 0.2941 | val loss 0.2381 | val acc 0.956`


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)

# --- 1. Synthetic data: two interleaved spirals, 2 classes ---
def make_spirals(n_per_class, noise=0.08):
    t = torch.linspace(0.5, 3 * torch.pi, n_per_class)
    xs, ys = [], []
    for label in (0, 1):
        r = t / (3 * torch.pi) * 2.5
        angle = t + label * torch.pi          # second spiral rotated 180 degrees
        pts = torch.stack([r * torch.cos(angle), r * torch.sin(angle)], dim=1)
        xs.append(pts + noise * torch.randn(n_per_class, 2))
        ys.append(torch.full((n_per_class,), label, dtype=torch.long))
    return torch.cat(xs), torch.cat(ys)

X, y = make_spirals(400)
perm = torch.randperm(len(X))                  # shuffle before splitting
X, y = X[perm], y[perm]
n_train = int(0.8 * len(X))
train_ds = TensorDataset(X[:n_train], y[:n_train])
val_ds = TensorDataset(X[n_train:], y[n_train:])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)

# --- 2. Model ---
class SpiralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)          # raw logits; CrossEntropyLoss applies softmax

model = SpiralNet()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# --- 3. The canonical loop ---
def evaluate():
    model.eval()                    # dropout off, etc.
    correct = n = 0
    total_loss = 0.0
    with torch.no_grad():           # no graph needed for evaluation
        for xb, yb in val_loader:
            logits = model(xb)
            total_loss += loss_fn(logits, yb).item() * len(xb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            n += len(xb)
    return total_loss / n, correct / n

for epoch in range(1, 151):
    model.train()                   # dropout on
    running = 0.0
    for xb, yb in train_loader:
        opt.zero_grad()             # 1. clear old gradients
        loss = loss_fn(model(xb), yb)   # 2. forward
        loss.backward()             # 3. backward
        opt.step()                  # 4. update
        running += loss.item() * len(xb)
    if epoch % 30 == 0:
        val_loss, val_acc = evaluate()
        print(f"epoch {epoch:3d}: train loss {running / n_train:.4f} | "
              f"val loss {val_loss:.4f} | val acc {val_acc:.3f}")

# --- 4. Save, reload into a fresh model, verify ---
ckpt = "spiralnet.pt"
torch.save(model.state_dict(), ckpt)

model = SpiralNet()                 # fresh, randomly initialized model
_, acc_fresh = evaluate()
model.load_state_dict(torch.load(ckpt))
_, acc_loaded = evaluate()
print(f"fresh model val acc: {acc_fresh:.3f} | after load_state_dict: {acc_loaded:.3f}")

## Reading the error message: shapes, dtypes, devices


*Expected output starts with:* `[shape] mat1 and mat2 shapes cannot be multiplied (4x8 and 10x3)`


In [ ]:
import warnings
import torch
import torch.nn as nn

torch.manual_seed(0)

# 1. Shape mismatch: the error names both shapes -- read them.
layer = nn.Linear(10, 3)
x = torch.randn(4, 8)                       # 8 features, layer expects 10
try:
    layer(x)
except RuntimeError as e:
    print(f"[shape] {e}")

# 2. Wrong label dtype at the loss.
logits = torch.randn(4, 3)
labels_float = torch.tensor([0.0, 1.0, 2.0, 1.0])   # should be torch.long
try:
    nn.CrossEntropyLoss()(logits, labels_float)
except RuntimeError as e:
    print(f"[dtype] {str(e)[:80]}")

# 3. The silent one: broadcasting turns (8,) vs (8,1) into an (8,8) comparison.
pred = torch.randn(8)
target = torch.randn(8, 1)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    wrong = nn.MSELoss()(pred, target)      # no error -- but not what you meant
right = nn.MSELoss()(pred, target.squeeze(1))
print(f"[broadcast] warning issued: {'target size' in str(caught[0].message)}")
print(f"[broadcast] loss with (8,) vs (8,1): {wrong.item():.4f}   after squeeze: {right.item():.4f}")
print(f"[broadcast] (pred - target).shape = {tuple((pred - target).shape)}  <- 64 pairwise differences")

## Reproducibility: seeds, determinism, and their limits


*Expected output starts with:* `seed 0, run 1: final batch loss = 0.6037580371`


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def train_once(seed):
    torch.manual_seed(seed)                  # governs init AND DataLoader shuffling
    torch.use_deterministic_algorithms(True) # error out on nondeterministic kernels
    gen = torch.Generator().manual_seed(0)   # data itself fixed across runs
    X = torch.randn(256, 4, generator=gen)
    y = (X.sum(dim=1) > 0).long()
    loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)
    model = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 2))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(5):
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
    return loss.item()

a, b, c = train_once(0), train_once(0), train_once(1)
print(f"seed 0, run 1: final batch loss = {a:.10f}")
print(f"seed 0, run 2: final batch loss = {b:.10f}")
print(f"seed 1:        final batch loss = {c:.10f}")
print(f"run 1 == run 2: {a == b}   (bitwise identical)")

## Try it yourself

1. Remove `model.eval()` from `evaluate()` in the end-to-end example and rerun. How much does reported validation accuracy change, and why does it now vary between calls on identical data?
2. Extend the checkpoint to a dictionary containing `model.state_dict()`, `opt.state_dict()`, and the epoch number. Stop training at epoch 75, restore everything, continue to 150, and compare the final validation loss to the uninterrupted run.
3. Replace `TensorDataset` with your own `Dataset` subclass implementing `__len__` and `__getitem__` that generates each spiral point on the fly from its index. Verify the training curve is unchanged when seeds are controlled.
4. Add a cosine schedule with warmup (from the [optimizers]({{ '/readings/ch1/optimizers/' | relative_url }}) section) to the end-to-end example and log `sched.get_last_lr()` at each printout. Does it change where the model reaches val acc 1.000?
5. Run the spiral training five times with seeds 0–4 (seeding both the data split and the model) and report the mean and range of final validation accuracy. Then introduce the `(8,)`-versus-`(8,1)` broadcasting bug from the error-message section into a regression variant of the loop and confirm it trains without any error while producing a wrong loss.


---

Full discussion of everything above: [S08 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s08/).
